<a href="https://colab.research.google.com/github/rohanyashraj/iai-workshop/blob/main/06_advanced_topics_and_case_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 · Advanced Topics in Agentic AI & Case Study Starter Pack

**Agentic AI for Health Actuaries** · IAI Seminar · 25 August 2026 · Hub: `github.com/rohanyashraj/iai-workshop`

> All data in this notebook is **hypothetical** — ABC Health is a fictional entity calibrated to plausible Indian health insurance experience, for teaching only.

**Mandatory group case study** · teams of ~5 · due **8 September 2026** · mentor office hours Tue & Thu.

Pick ONE track. Every track ships a **working, governed agent** plus the written governance pack.

| Track | Build | Start from |
|---|---|---|
| 1 · Model-tool agent | Wrap a GLM + XGBoost pipeline as governed tools; agent runs, compares, explains on a fresh dataset | Notebooks 02 + 05 |
| 2 · Document & RAG agent | Documentation agent grounded by RAG over a methodology pack, with logging + reviewer gate | Notebooks 01 + 05 |
| 3 · MCP automation agent | MCP + Claude Desktop automating one actuarial task end-to-end | Notebook 04 + MCP template |

**Deliverables:** (a) runnable notebook/repo · (b) spec + system prompt as governed artefacts · (c) guardrail demonstrated with a **before/after trace pair** · (d) the ten-question checklist answered in writing · (e) 1-page executive summary · (f) optional 5-min video.

**Evaluation:** runs 30% · governance & guardrails 30% · actuarial substance 25% · communication 15%.

## §0 · Advanced Topics in Agentic AI

**Used in:** Session 3, opening (bridges notebook 05's working Team example into the case-study tracks below). A little theory, a few small examples — not a full build. The worked example for all of this is notebook 05.

### A. Beyond one agent — orchestration patterns

One agent with several tools (notebooks 03-04) stops scaling once the tools serve genuinely different domains with different system prompts, different guardrails, or different personas — cramming PMI and IP logic into one prompt gets unwieldy fast, and it's harder to audit which rules govern which numbers. A **team** of specialist agents under a leader solves this, and Agno gives the leader four explicit delegation modes:

| Mode | What the leader does | When to reach for it |
|---|---|---|
| **Route** | Picks exactly one member (or answers directly with its own tools) and hands the whole query over | The query clearly belongs to one specialist — notebook 05's IP-vs-PMI-vs-policy-question split |
| **Coordinate** | Delegates pieces to multiple members, synthesises their outputs into one answer | A single request genuinely needs more than one specialist's output combined |
| **Broadcast** | Sends the same task to every member in parallel | You want several independent takes on the same question (e.g. a "second opinion" pattern) |
| **Tasks** | Decomposes work into an explicit ordered sequence | The work has real step-dependencies — step 2 can't start before step 1's output exists |

Notebook 05 uses **route** — the cleanest fit for "which product is this query about," and the mode most people mean when they say "orchestrator." The other three exist for genuinely different problem shapes; picking the wrong mode doesn't just under-perform, it produces a team that's harder to reason about and audit than a single well-scoped agent would have been.

### B. Grounding at scale — RAG as a governance tool

Notebook 05's `search_policy_docs` is a small, deliberate design choice worth naming explicitly: **numbers come from pricing tools, policy/assumption answers come from a citable document, and the two are never allowed to blend.** That separation is what makes RAG a *governance* mechanism here, not just a context-stuffing trick — an actuary reviewing a transcript can trace every policy claim back to a specific markdown section, the same way every price traces back to a specific tool call.

TF-IDF (what notebook 05 uses) is the right tool for a handful of short, topically distinct documents — no embeddings API call, no vector database, and it's transparent enough to debug by eye. It stops being the right tool once you have many long documents with real semantic overlap (two paragraphs that mean the same thing in different words, which TF-IDF's exact-term matching won't connect) — that's when embedding-based retrieval earns its added complexity. Match the retrieval technique to the actual document collection, not the other way around.

### C. Guardrails beyond a single tool

Every guardrail built across this series so far gates *one* tool's inputs — does this occupation exist, does this deferred period exist. At team scale, three more guardrail *shapes* start to matter:

- **Cross-agent consistency** — if a query genuinely needs both the IP and PMI agent's numbers combined, does anything check the two answers aren't contradicting each other before they're presented as one response?
- **Output-format guardrails** — notebook 05's "never use the `$` symbol" rule is a real example: a formatting failure mode that has nothing to do with inventing a number, but breaks the response just as badly.
- **Precision-of-meaning guardrails** — also already live in notebook 05's system prompts: correctly *citing* a real number while *mis-describing* what it measures (frequency labelled as claim probability) is a distinct failure mode from fabrication, and needs its own explicit rule, not just "never invent."

The pattern worth taking away: as a system scales from one tool to many agents, the guardrail surface doesn't grow linearly with the tool count — it grows with the number of places meaning can get lost in translation between tools, agents, and the final sentence a person reads.

### A brief word on evaluator/reviewer patterns

A common next step beyond a route-mode team is a **reviewer agent** — a second pass that checks a draft answer against the source tool outputs before it reaches the user, the same spirit as a second actuary reviewing a colleague's work before sign-off. It's a real, well-established pattern, deliberately **not built into notebook 05** — an addition worth prototyping yourself once the route-mode team above feels solid, not a prerequisite to it.

### A small example — sketching a "coordinate" alternative

Not a full build, just enough to see the shape. If a query genuinely needed *both* products priced and compared side by side, a **coordinate**-mode team (rather than notebook 05's route mode) would look like this:

```python
from agno.team import Team, TeamMode

# Same two specialist agents as notebook 05 (ip_agent, pmi_agent) - only the mode changes.
comparison_team = Team(
    name="ABC Health Comparison Desk",
    mode=TeamMode.coordinate,   # <- the only line that changed from notebook 05's route mode
    members=[ip_agent, pmi_agent],
    model=Gemini(id="gemini-3.5-flash-lite"),
    instructions=(
        "For any query comparing IP and PMI costs, delegate the IP pricing question to the "
        "IP Pricing Agent and the PMI pricing question to the PMI Pricing Agent, then "
        "synthesise both results into one side-by-side comparison. Never compute either "
        "product's premium yourself."
    ),
    markdown=True,
)
```

Notice what *didn't* change: both specialist agents, all their tools, all their guardrails. The delegation **mode** is a property of the team, not the members — this is exactly why Agno separates them, and why picking a mode is a design decision you can revisit without rebuilding your specialist agents from scratch.

### A small example — a cross-agent consistency guardrail

A minimal sketch of the "cross-agent consistency" guardrail shape named in §0.C — checking two independently-produced numbers agree on a shared fact (here, the age band) before presenting them together:

```python
def check_cross_agent_age_consistency(ip_result: dict, pmi_result: dict) -> dict:
    """Deterministic check - not an LLM judgement call. If two specialist agents both
    priced the same person, their age-derived facts should agree."""
    ip_age = ip_result.get('age')
    pmi_age = pmi_result.get('age')
    consistent = ip_age == pmi_age
    return {
        'consistent': consistent,
        'ip_age': ip_age,
        'pmi_age': pmi_age,
        'note': 'OK' if consistent else 'Mismatch - do not present these as the same person\'s quotes.'
    }
```

This is deliberately trivial — the point isn't the specific check, it's the *shape*: a plain Python function, gating a boundary between two agents' outputs, exactly the same "deterministic check, not a model judgement call" principle every guardrail in this series has followed since notebook 03.

## §1 · Your spec (fill this in FIRST — before any code)
> **Agent name:** …  
> **Who it serves:** (named persona + team)  
> **What it does:** (one sentence)  
> **Source(s) of truth:** (tables/documents the tools read)  
> **What it must NEVER do:** (these become system-prompt rules and guardrail tools)  
> **Output artefact:** (memo? report section? JSON?)  
> **Human gate:** (who reviews, before what action)

In [1]:
%pip install -q "agno==2.9.0" "google-genai==2.19.0" "xgboost==3.2.0" "shap==0.51.0" "statsmodels==0.14.6" "scikit-learn==1.9.0"

/Users/mohithsai/Downloads/Kasyap/IAI_25082026_notebooks/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
print("Hello")

Hello


In [3]:
import os
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("GOOGLE_API_KEY loaded from Colab Secrets.")
except Exception:
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
    print("Not running in Colab (or no secret set) - falling back to a local .env / environment variable.")
print("Ready. Team name:", "…")

Ready. Team name: …


## §2 · Datasets
Generate any of the three hypothetical ABC books below — **ABC Health** is today's primary book (same schema as notebooks 02 & 05); ABC Motor and ABC Life are there if your track fits them better — or bring a **public** dataset (never confidential data — reread the confidentiality slide).

In [4]:
# --- ABC Motor 2024: synthetic hypothetical dataset (alternative book, self-contained) ---
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
N = 50_000

motor = pd.DataFrame({
    "policy_id": [f"ABC-MOT-{i:06d}" for i in range(1, N + 1)],
    "vehicle_age_years": rng.integers(0, 16, N),
    "vehicle_make": rng.choice(["Maruti", "Hyundai", "Tata", "Mahindra", "Honda"], N,
                               p=[0.35, 0.25, 0.18, 0.12, 0.10]),
    "vehicle_segment": rng.choice(["Hatchback", "Sedan", "SUV", "MUV"], N,
                                  p=[0.45, 0.25, 0.22, 0.08]),
    "cubic_capacity": rng.choice([998, 1197, 1497, 1997, 2179], N),
    "ncb_pct": rng.choice([0, 20, 25, 35, 45, 50], N, p=[0.30, 0.15, 0.12, 0.15, 0.10, 0.18]),
    "policyholder_age": rng.integers(19, 75, N),
    "policyholder_gender": rng.choice(["M", "F"], N, p=[0.72, 0.28]),
    "region": rng.choice(["Tier1", "Tier2", "Tier3"], N, p=[0.40, 0.35, 0.25]),
    "prior_claims_3y": rng.choice([0, 1, 2, 3], N, p=[0.70, 0.20, 0.07, 0.03]),
})
motor["idv_inr"] = (900_000 * 0.9 ** motor["vehicle_age_years"]
                    * rng.uniform(0.8, 1.2, N)).round(-3)
# earned exposure over each policy's own observation year (mid-term entries/exits)
motor["exposure_years"] = rng.uniform(0.25, 1.0, N).round(3)
# underwriting cohort month — used for the out-of-time split (each cohort observed over its full policy year)
motor["inception_month"] = rng.integers(1, 13, N)

# True frequency model (the "world"): base 8% with realistic loadings
lin = (np.log(0.062)
       + 0.045 * motor["vehicle_age_years"]
       - 0.009 * motor["ncb_pct"]
       + 0.20 * motor["prior_claims_3y"]
       + np.where(motor["region"] == "Tier1", 0.12, np.where(motor["region"] == "Tier3", -0.10, 0.0))
       + np.where(motor["vehicle_segment"] == "SUV", 0.10, 0.0))
motor["claim_count"] = rng.poisson(np.exp(lin) * motor["exposure_years"])
# severity: Gamma, mean ~38k, only where claims exist
sev = rng.gamma(shape=2.0, scale=19_000, size=N)
motor["claim_amount_inr"] = (motor["claim_count"] * sev).round(0)

print("Shape:", motor.shape)
freq = motor.claim_count.sum() / motor.exposure_years.sum()
sev_mean = motor.loc[motor.claim_count > 0, "claim_amount_inr"].sum() / max(motor.claim_count.sum(), 1)
print(f"Portfolio frequency: {freq:.3f} per policy-year | mean severity: INR {sev_mean:,.0f}")
motor.head()

Shape: (50000, 15)
Portfolio frequency: 0.083 per policy-year | mean severity: INR 38,868


,policy_id,vehicle_age_years,vehicle_make,vehicle_segment,cubic_capacity,ncb_pct,policyholder_age,policyholder_gender,region,prior_claims_3y,idv_inr,exposure_years,inception_month,claim_count,claim_amount_inr
0,ABC-MOT-000001,1,Honda,MUV,1997,50,61,M,Tier1,0,684000.0,0.325,6,0,0.0
1,ABC-MOT-000002,12,Honda,Hatchback,1497,45,41,M,Tier1,0,248000.0,0.605,10,0,0.0
2,ABC-MOT-000003,10,Mahindra,SUV,1997,25,69,M,Tier2,0,278000.0,0.310,10,1,113626.0
3,ABC-MOT-000004,7,Hyundai,SUV,1497,35,40,M,Tier3,1,421000.0,0.969,7,0,0.0
4,ABC-MOT-000005,6,Hyundai,SUV,998,45,42,F,Tier1,0,506000.0,0.336,7,0,0.0


In [5]:
# --- ABC Health 2024: synthetic hypothetical PMI dataset (same schema as notebooks 02 & 05) ---
rng2 = np.random.default_rng(7)
M = 50_000
health = pd.DataFrame({
    "policy_id": [f"ABC-PMI-{i:06d}" for i in range(1, M + 1)],
    "member_age_years": rng2.integers(18, 76, M),
    "gender": rng2.choice(["M", "F"], M, p=[0.55, 0.45]),
    "plan_type": rng2.choice(["Individual", "Family Floater"], M, p=[0.55, 0.45]),
    "sum_insured_lakhs": rng2.choice([5, 10, 15, 25], M, p=[0.35, 0.35, 0.20, 0.10]),
    "bmi": np.clip(rng2.normal(25, 4, M), 16, 42).round(1),
    "city_tier": rng2.choice(["Tier1", "Tier2", "Tier3"], M, p=[0.40, 0.35, 0.25]),
    "ncb_pct": rng2.choice([0, 10, 20, 30], M, p=[0.30, 0.25, 0.20, 0.25]),
    "prior_claims_3y": rng2.choice([0, 1, 2, 3], M, p=[0.70, 0.20, 0.07, 0.03]),
})
health["exposure_years"] = rng2.uniform(0.25, 1.0, M).round(3)
health["inception_month"] = rng2.integers(1, 13, M)

lin_h = (np.log(0.0195)
         + 0.018 * health["member_age_years"]
         + 0.020 * health["sum_insured_lakhs"]
         + 0.030 * (health["bmi"] - 25)
         - 0.006 * health["ncb_pct"]
         + 0.150 * health["prior_claims_3y"]
         + np.where(health["city_tier"] == "Tier1", 0.10, np.where(health["city_tier"] == "Tier3", -0.12, 0.0))
         + np.where(health["plan_type"] == "Family Floater", 0.05, 0.0))
health["claim_count"] = rng2.poisson(np.exp(lin_h) * health["exposure_years"])
sev_h = rng2.gamma(shape=2.2, scale=38_600, size=M)
health["claim_amount_inr"] = (health["claim_count"] * sev_h).round(0)
print("ABC Health 2024:", health.shape)

# --- ABC Life — term cohort (condensed generator) ---
L = 20000
life = pd.DataFrame({
    "policy_id": [f"ABC-LIF-{i:06d}" for i in range(1, L + 1)],
    "issue_age": rng2.integers(25, 56, L),
    "smoker_status": rng2.choice(["NS", "S"], L, p=[0.85, 0.15]),
    "premium_frequency": rng2.choice(["Annual", "Monthly"], L, p=[0.55, 0.45]),
    "annualised_premium_inr": rng2.integers(8000, 60000, L),
    "region": rng2.choice(["Tier1", "Tier2", "Tier3"], L),
})
p_lapse = 0.06 + 0.05 * (life.premium_frequency == "Monthly") - 0.0006 * (life.issue_age - 40)
life["lapse_flag"] = rng2.binomial(1, p_lapse.clip(0.01, 0.9))
print("ABC Life cohort:", life.shape)


ABC Health 2024: (50000, 13)
ABC Life cohort: (20000, 7)


## §3 · Reusable scaffolds (copy, then adapt)
The guardrail pattern and the call logger — every track needs both.

In [6]:
# --- Guardrail template: gate a powerful tool behind a deterministic check ---
def make_existence_guardrail(loader, collection_key="factors"):
    """Returns a check function bound to YOUR source of truth (fixes the notebook-04 teaching bug)."""
    def check_in_source(name: str) -> dict:
        """MUST be called before explaining/using any item. Returns existence + valid list."""
        source = loader()
        return {"exists": name in source[collection_key],
                "valid": list(source[collection_key])}
    return check_in_source

# --- Run logger: checklist question 10 ---
import datetime, json, pathlib

def log_run(agent_name, user_query, trace, output, path="agent_run_log.jsonl"):
    rec = {"ts_utc": datetime.datetime.utcnow().isoformat(), "agent": agent_name,
           "query": user_query, "trace": trace, "output": output}
    with open(path, "a") as f:
        f.write(json.dumps(rec) + "\n")
    return rec["ts_utc"]

print("Scaffolds loaded.")

Scaffolds loaded.


## §4 · The ten-question checklist (submit answered, with evidence)
**Model (1–7)**
1. Purpose — what decision does this support, whose action does it trigger?
2. Data lineage — where did every column come from?
3. Train/test integrity — time-respecting split?
4. Fairness audit — protected AND proxy variables?
5. Interpretability — any single prediction in two sentences?
6. Monitoring — how are drift, decay, bias detected post-launch?
7. Sign-off — named, qualified owner?

**Agent (8–10)**
8. Tool scope — least data, least privilege, for every tool?
9. Guardrails — what mechanically stops hallucination, runaway loops, injection? Where is the human gate?
10. Trace — can you replay every tool call and every number's origin from the log?

## §5 · Submission checklist
- [ ] Runs end-to-end from a clean environment (`Runtime → Disconnect and delete runtime` → `Run all`)
- [ ] Spec (§1) filled and consistent with the system prompt
- [ ] Before/after guardrail trace pair included
- [ ] Ten questions answered with evidence
- [ ] 1-page executive summary (chief-actuary register)
- [ ] All data hypothetical or public; no confidential material anywhere, including prompts

*Office hours: Tue & Thu 6 PM IST on the hub. Bring your trace, not your slides.*